# Training NON-CoT (QLoRA SFT) — Skenario 3 arm non-CoT + base Skenario 1
### Self-contained: upload `train_sft.jsonl`

Latih 2 student: Qwen2.5-0.5B & Qwen2.5-1.5B (base). Target = cara + jawaban (\boxed).
**Setting:** GPU **T4 x2** (notebook pakai 1 GPU), Internet **On**, Add Input = `train_sft.jsonl`.

Output: LoRA adapter per model di panel Output. Nanti dieval pakai notebook Skenario 4 (tambah sebagai 'model kita').

### 1. Install

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

### 2. Konfigurasi

In [ ]:
import json, glob, os, gc, torch
hits = glob.glob('/kaggle/input/**/train_sft.jsonl', recursive=True)
TRAIN = hits[0] if hits else '/kaggle/input/train_sft.jsonl'
BASE_MODELS = {
    'qwen2.5-0.5b-noncot': 'Qwen/Qwen2.5-0.5B',
    'qwen2.5-1.5b-noncot': 'Qwen/Qwen2.5-1.5B',
}
EPOCHS = 1
MAX_SEQ = 2048
MAX_EXAMPLES = None   # set angka (mis 4000) buat uji cepat dulu
print('TRAIN =', TRAIN)

### 3. Load data SFT

In [ ]:
from datasets import Dataset
data = [json.loads(l) for l in open(TRAIN, encoding='utf-8') if l.strip()]
if MAX_EXAMPLES:
    data = data[:MAX_EXAMPLES]
print('contoh:', len(data), 'baris')
print(data[0]['messages'][1]['content'][:200])

### 4. Training loop (QLoRA 4-bit, LoRA adapter)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

for name, mid in BASE_MODELS.items():
    print('==='*10, name, mid)
    tok = AutoTokenizer.from_pretrained(mid)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    ds = Dataset.from_list(data).map(
        lambda ex: {'text': tok.apply_chat_template(ex['messages'], tokenize=False)},
        remove_columns=['messages'])

    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                             bnb_4bit_compute_dtype=torch.float16,
                             bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(mid, quantization_config=bnb,
                                                 device_map='auto')
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM',
                      target_modules=['q_proj','k_proj','v_proj','o_proj',
                                      'gate_proj','up_proj','down_proj'])
    out_dir = f'/kaggle/working/adapter_{name}'
    cfg = SFTConfig(output_dir=out_dir, num_train_epochs=EPOCHS,
                    per_device_train_batch_size=2, gradient_accumulation_steps=8,
                    learning_rate=2e-4, warmup_ratio=0.03, lr_scheduler_type='cosine',
                    fp16=True, logging_steps=20, save_strategy='epoch',
                    max_seq_length=MAX_SEQ, report_to='none', dataset_text_field='text')
    trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds, peft_config=lora)
    trainer.train()
    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    print('saved ->', out_dir)
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

### 5. Selesai
Adapter ada di `/kaggle/working/adapter_*`. Zip & download dari panel Output.
Eval: pakai notebook Skenario 4, load base + adapter sebagai 'model kita'.

In [ ]:
!cd /kaggle/working && zip -rq adapters.zip adapter_* && echo 'adapters.zip siap di Output'